# Text Side Ablation

Retrains all models with `use_text=True`. Results saved to `results/text_on/`.

Runs: 5 models × 2 datasets × 5 seeds = **50 new runs**.

In [ ]:
import sys
sys.path.insert(0, "..")

import json, os, warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch

from data.dataset import CascadeDataset
from models.gcn import GCNClassifier
from models.gat import GATClassifier
from models.bigcn import BiGCNClassifier
from models.gps import GPSClassifier
from models.cascade_gps import CascadeGPSClassifier
from training.trainer import run_experiment

DATA_ROOT = "../Twitter15_Twitter16"
TEXT_ON   = "../results/text_on"
SEEDS = [0, 1, 2, 3, 4]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
ds15 = CascadeDataset(root=DATA_ROOT, name="twitter15")
ds16 = CascadeDataset(root=DATA_ROOT, name="twitter16")
IN_CHANNELS = ds15[0].x.shape[1]
EDGE_DIM = ds15[0].edge_attr.shape[1]
print(f"Twitter15: {len(ds15)}  Twitter16: {len(ds16)}  in_channels={IN_CHANNELS}")

In [ ]:
kwargs = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, dropout=0.1, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"GCNClassifier+text — {ds_name}")
    run_experiment(
        GCNClassifier, kwargs, ds, SEEDS,
        epochs=200, warmup_ratio=0.1,
        device=DEVICE,
        out_path=f"{TEXT_ON}/gcn_{ds_name}.json",
    )


In [ ]:
kwargs = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, heads=4, dropout=0.1, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"GATClassifier+text — {ds_name}")
    run_experiment(
        GATClassifier, kwargs, ds, SEEDS,
        epochs=200, warmup_ratio=0.1,
        device=DEVICE,
        out_path=f"{TEXT_ON}/gat_{ds_name}.json",
    )


In [ ]:
kwargs = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, dropout=0.1, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"BiGCNClassifier+text — {ds_name}")
    run_experiment(
        BiGCNClassifier, kwargs, ds, SEEDS,
        epochs=200, warmup_ratio=0.1,
        device=DEVICE,
        out_path=f"{TEXT_ON}/bigcn_{ds_name}.json",
    )


In [ ]:
kwargs = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=4, heads=4, dropout=0.1, edge_dim=EDGE_DIM, readout='4way', use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"GPSClassifier+text — {ds_name}")
    run_experiment(
        GPSClassifier, kwargs, ds, SEEDS,
        epochs=200, lr=1e-3, weight_decay=0.05, patience=40, warmup_ratio=0.1, lap_pe_sign_flip=True, max_nodes_per_batch=8192,
        device=DEVICE,
        out_path=f"{TEXT_ON}/gps_4way_{ds_name}.json",
    )


In [ ]:
kwargs = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, heads=4, dropout=0.1, edge_dim=EDGE_DIM, use_text=True)

for ds, ds_name in ((ds15, "twitter15"), (ds16, "twitter16")):
    print("=" * 50, f"CascadeGPSClassifier+text — {ds_name}")
    run_experiment(
        CascadeGPSClassifier, kwargs, ds, SEEDS,
        epochs=200, lr=1e-3, weight_decay=0.05, patience=40, warmup_ratio=0.1, lap_pe_sign_flip=True, max_nodes_per_batch=2048,
        device=DEVICE,
        out_path=f"{TEXT_ON}/cascade_gps_{ds_name}.json",
        save_checkpoint=True,
    )


## Results


In [ ]:
import numpy as np
from analysis.stats import load_results, summarise

results = load_results("../results")

ALL_MODELS = ["gcn", "gat", "bigcn", "gps_4way", "cascade_gps"]

print("{:<14} {:<12} {:>14} {:>14} {:>8}".format("Model","Dataset","text=off","text=on","DeltaF1"))
print("-" * 66)
for model in ALL_MODELS:
    for dataset in ("twitter15", "twitter16"):
        off = results.get((model, dataset, False))
        on  = results.get((model, dataset, True))
        s_off = f"{summarise(off)['mean']:.3f} +/-{summarise(off)['std']:.3f}" if off else '-'
        s_on  = f"{summarise(on)['mean']:.3f} +/-{summarise(on)['std']:.3f}" if on else '-'
        delta = f"{summarise(on)['mean'] - summarise(off)['mean']:+.3f}" if off and on else '-'
        print(f"{model:<14} {dataset:<12} {s_off:>14} {s_on:>14} {delta:>8}")
